In [1]:
import numpy as np
import pandas as pd

In [2]:
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')

In [3]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [5]:
movies = movies.merge(credits, on='title')

In [6]:
# genres , id, keywords, title, overview, cast, crew

movies = movies[['movie_id', 'keywords', 'title', 'overview', 'cast', 'crew', 'genres']]

In [7]:
import ast
import pandas as pd

TARGET_COL = 'genres'

def clean_genres(cell):
    if pd.isna(cell):
        return []
    try:
        items = ast.literal_eval(cell) if isinstance(cell, str) else cell
    except Exception:
        return []
    cleaned = []
    for g in items:
        if isinstance(g, dict) and 'name' in g:
            cleaned.append({'name': g['name']})
    return cleaned


# optionally: get just the list of names
movies['genres'] = movies[TARGET_COL].apply(clean_genres).apply(lambda lst: [d['name'] for d in lst])

movies[['genres']].head(10)

,genres
0,"[Action, Adventure, Fantasy, Science Fiction]"
1,"[Adventure, Fantasy, Action]"
2,"[Action, Adventure, Crime]"
3,"[Action, Crime, Drama, Thriller]"
4,"[Action, Adventure, Science Fiction]"
5,"[Fantasy, Action, Adventure]"
6,"[Animation, Family]"
7,"[Action, Adventure, Science Fiction]"
8,"[Adventure, Fantasy, Family]"
9,"[Action, Adventure, Fantasy]"


In [8]:
import ast

def extract_top_cast(cell, n=3):
    if pd.isna(cell):
        return []
    try:
        items = ast.literal_eval(cell) if isinstance(cell, str) else cell
    except Exception:
        return []
    names = []
    for c in items:
        if isinstance(c, dict) and 'name' in c:
            names.append(c['name'])
            if len(names) >= n:
                break
    return names

# Replace 'cast' with top 3 cast names
movies['cast'] = movies['cast'].apply(lambda cell: extract_top_cast(cell, 3))

movies[['title','cast']].head(10)

,title,cast
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]"
3,The Dark Knight Rises,"[Christian Bale, Michael Caine, Gary Oldman]"
4,John Carter,"[Taylor Kitsch, Lynn Collins, Samantha Morton]"
5,Spider-Man 3,"[Tobey Maguire, Kirsten Dunst, James Franco]"
6,Tangled,"[Zachary Levi, Mandy Moore, Donna Murphy]"
7,Avengers: Age of Ultron,"[Robert Downey Jr., Chris Hemsworth, Mark Ruff..."
8,Harry Potter and the Half-Blood Prince,"[Daniel Radcliffe, Rupert Grint, Emma Watson]"
9,Batman v Superman: Dawn of Justice,"[Ben Affleck, Henry Cavill, Gal Gadot]"


In [9]:
import ast

def extract_director_from_crew(cell):
    if pd.isna(cell):
        return None
    try:
        items = ast.literal_eval(cell) if isinstance(cell, str) else cell
    except Exception:
        return None
    for p in items:
        if isinstance(p, dict) and p.get('job') == 'Director':
            return p.get('name')
    return None

# Prefer crew column for director
if 'crew' in movies.columns:
    movies['crew'] = movies['crew'].apply(extract_director_from_crew)
else:
    movies['crew'] = movies['cast'].apply(extract_director_from_crew)

movies[['title','crew']].head(10)


,title,crew
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton
5,Spider-Man 3,Sam Raimi
6,Tangled,Byron Howard
7,Avengers: Age of Ultron,Joss Whedon
8,Harry Potter and the Half-Blood Prince,David Yates
9,Batman v Superman: Dawn of Justice,Zack Snyder


In [10]:
movies.head()

,movie_id,keywords,title,overview,cast,crew,genres
0,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",Avatar,"In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[Action, Adventure, Fantasy, Science Fiction]"
1,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[Adventure, Fantasy, Action]"
2,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",Spectre,A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[Action, Adventure, Crime]"
3,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",The Dark Knight Rises,Following the death of District Attorney Harve...,"[Christian Bale, Michael Caine, Gary Oldman]",Christopher Nolan,"[Action, Crime, Drama, Thriller]"
4,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",John Carter,"John Carter is a war-weary, former military ca...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",Andrew Stanton,"[Action, Adventure, Science Fiction]"


In [11]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4809 non-null   int64 
 1   keywords  4809 non-null   object
 2   title     4809 non-null   object
 3   overview  4806 non-null   object
 4   cast      4809 non-null   object
 5   crew      4779 non-null   object
 6   genres    4809 non-null   object
dtypes: int64(1), object(6)
memory usage: 263.1+ KB


In [12]:
movies['overview'] = movies['overview'].fillna('').astype(str).apply(lambda x: x.split())

In [13]:
def clean_keywords(cell):
    if pd.isna(cell):
        return []
    try:
        items = ast.literal_eval(cell) if isinstance(cell, str) else cell
    except Exception:
        return []
    return [item['name'] for item in items if isinstance(item, dict) and 'name' in item]

movies['keywords'] = movies['keywords'].apply(clean_keywords)

movies[['title', 'keywords']].head(10) 

,title,keywords
0,Avatar,"[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"[ocean, drug abuse, exotic island, east india ..."
2,Spectre,"[spy, based on novel, secret agent, sequel, mi..."
3,The Dark Knight Rises,"[dc comics, crime fighter, terrorist, secret i..."
4,John Carter,"[based on novel, mars, medallion, space travel..."
5,Spider-Man 3,"[dual identity, amnesia, sandstorm, love of on..."
6,Tangled,"[hostage, magic, horse, fairy tale, musical, p..."
7,Avengers: Age of Ultron,"[marvel comic, sequel, superhero, based on com..."
8,Harry Potter and the Half-Blood Prince,"[witch, magic, broom, school of witchcraft, wi..."
9,Batman v Superman: Dawn of Justice,"[dc comics, vigilante, superhero, based on com..."


In [14]:
movies.head()

,movie_id,keywords,title,overview,cast,crew,genres
0,19995,"[culture clash, future, space war, space colon...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[Action, Adventure, Fantasy, Science Fiction]"
1,285,"[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[Adventure, Fantasy, Action]"
2,206647,"[spy, based on novel, secret agent, sequel, mi...",Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[Action, Adventure, Crime]"
3,49026,"[dc comics, crime fighter, terrorist, secret i...",The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Christian Bale, Michael Caine, Gary Oldman]",Christopher Nolan,"[Action, Crime, Drama, Thriller]"
4,49529,"[based on novel, mars, medallion, space travel...",John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",Andrew Stanton,"[Action, Adventure, Science Fiction]"


In [15]:
cols_list = ['keywords', 'cast', 'genres']
for col in cols_list:
    if col in movies.columns:
        movies[col] = movies[col].apply(lambda lst: [s.replace(' ', '') for s in lst] if isinstance(lst, list) else lst)

if 'crew' in movies.columns:
    movies['crew'] = movies['crew'].apply(lambda s: s.replace(' ', '') if isinstance(s, str) else s)

movies[['keywords','cast','genres','crew']].head()

,keywords,cast,genres,crew
0,"[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]","[Action, Adventure, Fantasy, ScienceFiction]",JamesCameron
1,"[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]","[Adventure, Fantasy, Action]",GoreVerbinski
2,"[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]","[Action, Adventure, Crime]",SamMendes
3,"[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]","[Action, Crime, Drama, Thriller]",ChristopherNolan
4,"[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]","[Action, Adventure, ScienceFiction]",AndrewStanton


In [16]:
movies.head()

,movie_id,keywords,title,overview,cast,crew,genres
0,19995,"[cultureclash, future, spacewar, spacecolony, ...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",JamesCameron,"[Action, Adventure, Fantasy, ScienceFiction]"
1,285,"[ocean, drugabuse, exoticisland, eastindiatrad...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",GoreVerbinski,"[Adventure, Fantasy, Action]"
2,206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...",Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[DanielCraig, ChristophWaltz, LéaSeydoux]",SamMendes,"[Action, Adventure, Crime]"
3,49026,"[dccomics, crimefighter, terrorist, secretiden...",The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[ChristianBale, MichaelCaine, GaryOldman]",ChristopherNolan,"[Action, Crime, Drama, Thriller]"
4,49529,"[basedonnovel, mars, medallion, spacetravel, p...",John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[TaylorKitsch, LynnCollins, SamanthaMorton]",AndrewStanton,"[Action, Adventure, ScienceFiction]"


In [17]:
movies['overview'] = movies['overview'].apply(lambda x: x if isinstance(x, list) else x.split())

In [18]:
# ensure 'crew' is a list (not a string) so concatenation with other lists works
movies['crew'] = movies['crew'].apply(lambda x: [x] if isinstance(x, str) else ([] if pd.isna(x) else x))

# build tags by concatenating lists
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [19]:
new = movies.drop(columns=['overview','genres','keywords','cast','crew'])

In [20]:
new['tags'] = new['tags'].apply(lambda x: " ".join(x))
new.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [21]:
new['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [22]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# Using TF-IDF instead of CountVectorizer for better feature representation
# TF-IDF gives higher weight to distinctive words and lower weight to common words
cv = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))

In [23]:
vector = cv.fit_transform(new['tags']).toarray()

In [24]:
vector.shape

(4809, 5000)

In [25]:
# cosine simalarity is used to calculate the distance between the vector points, we are not calculating using eucledian distance
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
similarity = cosine_similarity(vector)

In [27]:
similarity

array([[1.        , 0.04289272, 0.02449826, ..., 0.00541477, 0.00588179,
        0.        ],
       [0.04289272, 1.        , 0.01233614, ..., 0.01798152, 0.        ,
        0.        ],
       [0.02449826, 0.01233614, 1.        , ..., 0.01639946, 0.        ,
        0.        ],
       ...,
       [0.00541477, 0.01798152, 0.01639946, ..., 1.        , 0.02977675,
        0.03276643],
       [0.00588179, 0.        , 0.        , ..., 0.02977675, 1.        ,
        0.01649133],
       [0.        , 0.        , 0.        , ..., 0.03276643, 0.01649133,
        1.        ]])

In [28]:
new[new['title'] == 'The Lego Movie'].index[0]

744

In [29]:
import pickle

In [32]:
pickle.dump(movies,open('movie_list.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))